In [0]:
import importlib
import sys
from pathlib import Path
from pyspark.sql import functions as F
project_root = str(Path.cwd().resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.storage_config as storage_config
from pyspark.sql import functions as F
importlib.reload(storage_config)
storage_config.spark = spark
storage_config.configure_storage()

In [0]:
df = (
    spark.read
    .format("delta")
    .load("abfss://bronze@secondstorage89.dfs.core.windows.net/payments/")
)

In [0]:
df.show(10)

```md
  Column                 | Meaning                                      |
| ---------------------- | -------------------------------------------- |
| `order_id`             | payment of which order                       |
| `payment_sequential`   | seq of the payment in this order             |
| `payment_type`         | method of payment                            |
| `payment_installments` | how may installments are there               |
| `payment_value`        | amount of the payment                        |
| `_ingested_at`         | Bronze ingestion timestamp                   |
| `_source_file`         | Source file/path                             |

```
```md
payment_sequential = 1
→ first payment record

payment_sequential = 2
→ second payment record
```

payment seq = A sequence number that distinguishes multiple payment entries belonging to the same order.
```md
order_id    payment_sequential    payment_type    payment_value
O001        1                     credit_card     100.00
O001        2                     voucher          20.00
```

In [0]:
df.printSchema()

In [0]:
df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

In [0]:
df.groupBy(
    "order_id",
    "payment_sequential"
).count().filter(
    F.col("count") > 1
).show()

In [0]:
df.groupBy("payment_type").count().show()

In [0]:
df.filter(
    F.col("payment_type") == "not_defined"
).show(truncate=False)

In [0]:
invalid_payment = (
    (F.col("payment_type") == "not_defined") &
    (F.col("payment_value") == 0)
)

In [0]:
quarantine_payment = df.filter(invalid_payment)
silver_payment = df.filter(~invalid_payment)


In [0]:
silver_payment.show(10)

In [0]:
silver_payment.select("payment_value", "payment_installments").describe().show()


In [0]:
df.write.mode("overwrite").save("abfss://silver@secondstorage89.dfs.core.windows.net/payments/")